# EcoSort AI — application integration and prediction API (Notebook 04)

**Purpose:** turn the trained model from Notebook 03 into reusable application code and a FastAPI service.

Run notebooks 01–03 first. This notebook:
1. Finds and loads the saved `.keras` model and metadata.
2. Creates reusable prediction code in `src/predictor.py`.
3. Creates a configurable disposal-rule file (kept separate from image classification).
4. Creates a FastAPI app with health, single-image and bulk-image endpoints.
5. Runs local smoke tests against the saved model.

**Important:** this is an API development notebook, not cloud deployment. Do not expose Colab's temporary runtime as a production service. Docker, deployment, dashboard and load testing come later.

## 1. Environment and project paths

Run in the same Colab environment/Drive used by Notebook 03. The model file and `model_metadata.json` must exist.

In [2]:
from pathlib import Path
import os, json, sys, subprocess, textwrap

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/ecosort")
else:
    PROJECT_DIR = Path.cwd()

MODELS_DIR = PROJECT_DIR / "models"
SRC_DIR = PROJECT_DIR / "src"
REPORTS_DIR = PROJECT_DIR / "reports"
SRC_DIR.mkdir(parents=True, exist_ok=True)

metadata_path = MODELS_DIR / "model_metadata.json"
assert metadata_path.exists(), f"Missing {metadata_path}. Run Notebook 03 first."
metadata = json.loads(metadata_path.read_text())
MODEL_PATH = MODELS_DIR / metadata["model_file"]
assert MODEL_PATH.exists(), f"Missing trained model: {MODEL_PATH}"

print("Project:", PROJECT_DIR)
print("Model:", MODEL_PATH)
print("Version:", metadata["model_version"])
print("Classes:", metadata["class_names"])

Mounted at /content/drive
Project: /content/drive/MyDrive/ecosort
Model: /content/drive/MyDrive/ecosort/models/ecosort_mobilenetv2_v1.keras
Version: v1
Classes: ['Cardboard', 'Food Organics', 'Glass', 'Metal', 'Miscellaneous Trash', 'Paper', 'Plastic', 'Textile Trash', 'Vegetation']


## 2. Install API dependencies

TensorFlow should already be installed from Notebook 03. FastAPI and Uvicorn provide the HTTP API; Pillow validates/decodes uploaded images.

In [3]:
%pip -q install fastapi uvicorn python-multipart pillow httpx

## 3. Create configurable disposal rules

The model predicts a **material class**. A separate rules file maps that class to a disposal stream. This separation lets EcoSort adapt to local waste-management rules without retraining the image classifier. The example values below are placeholders for development; verify the actual accepted streams and local rules before presenting them as official Kigali/Rwanda guidance.

In [4]:
disposal_rules = {
    "rules_version": "draft-1",
    "note": "Illustrative mapping only. Confirm local collection rules before operational use.",
    "streams": {
        "Cardboard": "recyclable",
        "Food Organics": "biodegradable",
        "Glass": "recyclable",
        "Metal": "recyclable",
        "Miscellaneous Trash": "residual",
        "Paper": "recyclable",
        "Plastic": "recyclable",
        "Textile Trash": "residual",
        "Vegetation": "biodegradable"
    },
    "manual_review_classes": ["Miscellaneous Trash"]
}
rules_path = PROJECT_DIR / "disposal_rules.json"
rules_path.write_text(json.dumps(disposal_rules, indent=2))
print("Wrote:", rules_path)

Wrote: /content/drive/MyDrive/ecosort/disposal_rules.json


## 4. Write reusable prediction module

This module loads the exact model artifact and metadata, validates image files, preprocesses images to 224×224 RGB, and returns top candidates plus an uncertainty flag. It accepts raw 0–255 pixels because the trained model contains its own MobileNetV2 scaling layer.

In [5]:
predictor_source = r"""
import json
from pathlib import Path
from typing import Any
import numpy as np
import tensorflow as tf
from PIL import Image, UnidentifiedImageError

class EcoSortPredictor:
    def __init__(self, model_path: str, metadata_path: str, rules_path: str | None = None):
        self.model_path = Path(model_path)
        self.metadata_path = Path(metadata_path)
        self.metadata = json.loads(self.metadata_path.read_text())
        self.model = tf.keras.models.load_model(self.model_path)
        self.class_names = self.metadata["class_names"]
        self.image_size = tuple(self.metadata.get("input_size", [224, 224, 3])[:2])
        self.threshold = float(self.metadata.get("confidence_threshold", 0.70))
        self.rules = {}
        if rules_path and Path(rules_path).exists():
            self.rules = json.loads(Path(rules_path).read_text())

    def prepare_image(self, image_file):
        try:
            with Image.open(image_file) as image:
                image = image.convert("RGB").resize(
                    (self.image_size[1], self.image_size[0])
                )
                array = np.asarray(image, dtype=np.float32)
        except (UnidentifiedImageError, OSError, ValueError) as exc:
            raise ValueError("File is not a valid readable image.") from exc
        return np.expand_dims(array, axis=0)

    def predict(self, image_file, top_k: int = 3) -> dict[str, Any]:
        if not 1 <= top_k <= len(self.class_names):
            raise ValueError(f"top_k must be between 1 and {len(self.class_names)}")
        batch = self.prepare_image(image_file)
        probabilities = self.model.predict(batch, verbose=0)[0]
        indices = np.argsort(probabilities)[::-1][:top_k]
        top = [
            {"class_name": self.class_names[int(i)],
             "confidence": round(float(probabilities[i]), 4)}
            for i in indices
        ]
        best = top[0]
        uncertain = best["confidence"] < self.threshold
        stream = None
        if not uncertain:
            stream = self.rules.get("streams", {}).get(best["class_name"])
        manual_review = (
            uncertain or best["class_name"] in self.rules.get("manual_review_classes", [])
        )
        return {
            "predicted_class": best["class_name"],
            "confidence": best["confidence"],
            "uncertain": uncertain,
            "top_k": top,
            "disposal_stream": None if manual_review else stream,
            "manual_review": manual_review,
            "model_version": self.metadata["model_version"]
        }
"""
(SRC_DIR / "predictor.py").write_text(textwrap.dedent(predictor_source).strip() + "\n")
print("Created", SRC_DIR / "predictor.py")

Created /content/drive/MyDrive/ecosort/src/predictor.py


## 5. Create the FastAPI application

Endpoints:
- `GET /health` — service/model status and model version.
- `POST /predict` — one image.
- `POST /predict/batch` — multiple images in one request.

Uploads are checked by the image decoder; the API does not trust the filename extension alone.

In [6]:
app_source = r"""
import os
import json
from pathlib import Path
from typing import List
from fastapi import FastAPI, File, UploadFile, HTTPException
from src.predictor import EcoSortPredictor

ROOT = Path(os.getenv("ECOSORT_PROJECT_DIR", Path(__file__).resolve().parent.parent))
MODEL_DIR = ROOT / "models"
predictor = EcoSortPredictor(
    model_path=str(MODEL_DIR / json.loads((MODEL_DIR / "model_metadata.json").read_text())["model_file"]),
    metadata_path=str(MODEL_DIR / "model_metadata.json"),
    rules_path=str(ROOT / "disposal_rules.json"),
)
app = FastAPI(title="EcoSort AI API", version="1.0.0",
              description="Waste material classification API")

async def run_prediction(upload: UploadFile):
    content = await upload.read()
    if not content:
        raise HTTPException(status_code=400, detail="Uploaded file is empty.")
    try:
        from io import BytesIO
        return predictor.predict(BytesIO(content))
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc))
    finally:
        await upload.close()

@app.get("/health")
def health():
    return {
        "status": "ok",
        "model_loaded": predictor.model is not None,
        "model_version": predictor.metadata["model_version"]
    }

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    result = await run_prediction(file)
    return {"filename": file.filename, **result}

@app.post("/predict/batch")
async def predict_batch(files: List[UploadFile] = File(...)):
    if not files:
        raise HTTPException(status_code=400, detail="Upload at least one image.")
    results = []
    for upload in files:
        try:
            result = await run_prediction(upload)
            results.append({"filename": upload.filename, **result})
        except HTTPException as exc:
            results.append({"filename": upload.filename, "error": exc.detail})
    return {"count": len(results), "results": results}
"""
(SRC_DIR / "app.py").write_text(textwrap.dedent(app_source).strip() + "\n")
print("Created", SRC_DIR / "app.py")

Created /content/drive/MyDrive/ecosort/src/app.py


## 6. Write requirements and inspect project files

In [7]:
requirements = """fastapi
uvicorn[standard]
python-multipart
pillow
numpy
tensorflow
"""
(PROJECT_DIR / "requirements-api.txt").write_text(requirements)
for path in [SRC_DIR / "predictor.py", SRC_DIR / "app.py", rules_path, PROJECT_DIR / "requirements-api.txt"]:
    print(path.relative_to(PROJECT_DIR), "-", path.stat().st_size, "bytes")

src/predictor.py - 2571 bytes
src/app.py - 1948 bytes
disposal_rules.json - 492 bytes
requirements-api.txt - 67 bytes


## 7. Load predictor and test a real image

The test uses an image from the held-out test split only as a smoke test of the serving code. It does not change model weights or tune the model. Notebook 03 remains the source of the reported test metrics.

In [8]:
import pandas as pd
sys.path.insert(0, str(PROJECT_DIR))
(SRC_DIR / "__init__.py").write_text("")
from src.predictor import EcoSortPredictor

predictor = EcoSortPredictor(
    str(MODEL_PATH), str(metadata_path), str(rules_path)
)
test_csv = PROJECT_DIR / "splits" / "test.csv"
assert test_csv.exists(), "Missing test.csv; run Notebook 02."
test_df = pd.read_csv(test_csv)
# Notebook 03 split CSV paths are relative to the RealWaste image root.
raw_dir = PROJECT_DIR / "data_raw"
realwaste_root = next(p for p in raw_dir.rglob("RealWaste") if p.is_dir() and any(x.is_dir() for x in p.iterdir()))
sample_path = realwaste_root / test_df.iloc[0]["path"]
assert sample_path.exists(), sample_path
result = predictor.predict(sample_path)
print("Image:", sample_path.name)
print(json.dumps(result, indent=2))

Image: Cardboard_105.jpg
{
  "predicted_class": "Plastic",
  "confidence": 0.5571,
  "uncertain": true,
  "top_k": [
    {
      "class_name": "Plastic",
      "confidence": 0.5571
    },
    {
      "class_name": "Textile Trash",
      "confidence": 0.2089
    },
    {
      "class_name": "Miscellaneous Trash",
      "confidence": 0.0973
    }
  ],
  "disposal_stream": null,
  "manual_review": true,
  "model_version": "v1"
}


## 8. Test the HTTP endpoints in-process

FastAPI's TestClient exercises the same routes without requiring a public server. This is a local integration smoke test, not a load test.

In [9]:
from fastapi.testclient import TestClient
from src.app import app

client = TestClient(app)
health_response = client.get("/health")
print("GET /health:", health_response.status_code, health_response.json())
assert health_response.status_code == 200
assert health_response.json()["model_loaded"] is True

with open(sample_path, "rb") as f:
    single_response = client.post(
        "/predict",
        files={"file": (sample_path.name, f, "image/jpeg")}
    )
print("POST /predict:", single_response.status_code)
print(json.dumps(single_response.json(), indent=2))
assert single_response.status_code == 200

GET /health: 200 {'status': 'ok', 'model_loaded': True, 'model_version': 'v1'}
POST /predict: 200
{
  "filename": "Cardboard_105.jpg",
  "predicted_class": "Plastic",
  "confidence": 0.5571,
  "uncertain": true,
  "top_k": [
    {
      "class_name": "Plastic",
      "confidence": 0.5571
    },
    {
      "class_name": "Textile Trash",
      "confidence": 0.2089
    },
    {
      "class_name": "Miscellaneous Trash",
      "confidence": 0.0973
    }
  ],
  "disposal_stream": null,
  "manual_review": true,
  "model_version": "v1"
}


## 9. Test bulk prediction and invalid uploads

In [10]:
with open(sample_path, "rb") as f1, open(sample_path, "rb") as f2:
    batch_response = client.post(
        "/predict/batch",
        files=[
            ("files", (sample_path.name, f1, "image/jpeg")),
            ("files", ("second_" + sample_path.name, f2, "image/jpeg"))
        ]
    )
print("POST /predict/batch:", batch_response.status_code)
print(batch_response.json())
assert batch_response.status_code == 200
assert batch_response.json()["count"] == 2

bad_response = client.post(
    "/predict",
    files={"file": ("not_an_image.txt", b"this is not an image", "text/plain")}
)
print("Invalid upload:", bad_response.status_code, bad_response.json())
assert bad_response.status_code == 400

POST /predict/batch: 200
{'count': 2, 'results': [{'filename': 'Cardboard_105.jpg', 'predicted_class': 'Plastic', 'confidence': 0.5571, 'uncertain': True, 'top_k': [{'class_name': 'Plastic', 'confidence': 0.5571}, {'class_name': 'Textile Trash', 'confidence': 0.2089}, {'class_name': 'Miscellaneous Trash', 'confidence': 0.0973}], 'disposal_stream': None, 'manual_review': True, 'model_version': 'v1'}, {'filename': 'second_Cardboard_105.jpg', 'predicted_class': 'Plastic', 'confidence': 0.5571, 'uncertain': True, 'top_k': [{'class_name': 'Plastic', 'confidence': 0.5571}, {'class_name': 'Textile Trash', 'confidence': 0.2089}, {'class_name': 'Miscellaneous Trash', 'confidence': 0.0973}], 'disposal_stream': None, 'manual_review': True, 'model_version': 'v1'}]}
Invalid upload: 400 {'detail': 'File is not a valid readable image.'}


## 10. Run the API locally (optional)

In a local clone of the project, install `requirements-api.txt`, then run the command below from the project root. In Colab, use the in-process tests above; the runtime is temporary and should not be treated as a permanent public API.

In [11]:
print("From the EcoSort project root, run:")
print("  pip install -r requirements-api.txt")
print("  uvicorn src.app:app --host 127.0.0.1 --port 8000")
print("Then open:")
print("  http://127.0.0.1:8000/docs")

From the EcoSort project root, run:
  pip install -r requirements-api.txt
  uvicorn src.app:app --host 127.0.0.1 --port 8000
Then open:
  http://127.0.0.1:8000/docs


## 11. What this notebook has and has not completed

**Completed here:** reusable inference module, model/metadata loading, health endpoint, single-image endpoint, batch-image endpoint, confidence flag, configurable disposal mapping, and endpoint smoke tests.

**Not completed here:** authenticated access, request-size limits/rate limiting, persistent upload storage, asynchronous/background retraining, model registry/atomic promotion, Docker image, cloud deployment, dashboard, and Locust performance testing. These are later stages. Do not describe them as implemented until tested.

**Important design note:** the API currently loads the model version named in the metadata's model file. If you change the model version in Notebook 03, update the API loader to read `metadata['model_file']` rather than hardcoding `v1`; this is a small improvement to make before deployment.